# 02 — Pré-processamento

**Issues #5 e #6** · Milestone M2

> Notebook **autocontido** — baixa o CSV se necessário (igual ao `01_EDA`).

## Regras
- `time_left >= 150`, 5v5, excluir `de_cache`
- Features derivadas: `money_diff`, buy tier (média $/jogador), AWP, rifles, util
- Pipeline: `StandardScaler` + `OneHotEncoder` + split 80/20 estratificado

In [ ]:
import json
import shutil
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

DATA_RAW = Path("/content/data/raw")
DATA_PROCESSED = Path("/content/data/processed")
CSV_DEFAULT = DATA_RAW / "csgo_round_snapshots.csv"
DATASET = "christianlillelund/csgo-round-winner-classification"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


def ensure_dataset() -> Path:
    if CSV_DEFAULT.exists():
        return CSV_DEFAULT
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    cache_path = Path(kagglehub.dataset_download(DATASET))
    for src in cache_path.rglob("*.csv"):
        shutil.copy2(src, DATA_RAW / src.name)
    if not CSV_DEFAULT.exists():
        raise FileNotFoundError(cache_path)
    return CSV_DEFAULT


csv_path = ensure_dataset()
print("CSV:", csv_path)

## 1. Filtros e feature engineering (issue #5)

In [ ]:
MAP_EXCLUDE = {"de_cache"}

RIFLE_COLS_CT = [
    "ct_weapon_ak47", "ct_weapon_m4a4", "ct_weapon_m4a1s",
    "ct_weapon_galilar", "ct_weapon_famas", "ct_weapon_sg553", "ct_weapon_aug",
]
RIFLE_COLS_T = [c.replace("ct_", "t_") for c in RIFLE_COLS_CT]

UTIL_COLS_CT = [
    "ct_grenade_hegrenade", "ct_grenade_flashbang", "ct_grenade_smokegrenade",
    "ct_grenade_incendiarygrenade", "ct_grenade_molotovgrenade", "ct_grenade_decoygrenade",
]
UTIL_COLS_T = [c.replace("ct_", "t_") for c in UTIL_COLS_CT]


BUY_TIER_ORDER = ["force", "eco", "mixed", "half", "full"]


def buy_tier(team_money: float) -> str:
    """Fase de compra a partir do dinheiro total do time (soma dos 5 jogadores).

    Proxy: média por jogador (team_money / 5). Faixas CS:GO competitivo:
    force < $1.700, eco < $2.000, half $2.800–$3.799, full >= $4.700, mixed nas transições.
    Saldo mínimo pós-compra (half) e loss bonus não estão no dataset.
    """
    m = team_money / 5
    if m >= 4700:
        return "full"
    if 2800 <= m < 3800:
        return "half"
    if m < 1700:
        return "force"
    if m < 2000:
        return "eco"
    return "mixed"


def filter_round_start(df: pd.DataFrame) -> pd.DataFrame:
    mask = df["time_left"] >= 150
    mask &= df["ct_players_alive"] == 5
    mask &= df["t_players_alive"] == 5
    mask &= ~df["map"].isin(MAP_EXCLUDE)
    return df.loc[mask].copy()


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    new_cols = pd.DataFrame(
        {
            "money_diff": out["ct_money"] - out["t_money"],
            "health_diff": out["ct_health"] - out["t_health"],
            "armor_diff": out["ct_armor"] - out["t_armor"],
            "players_alive_diff": out["ct_players_alive"] - out["t_players_alive"],
            "money_ratio": out["ct_money"] / out["t_money"].replace(0, 1),
            "ct_has_awp": (out["ct_weapon_awp"] > 0).astype(int),
            "t_has_awp": (out["t_weapon_awp"] > 0).astype(int),
            "ct_rifle_count": out[RIFLE_COLS_CT].sum(axis=1),
            "t_rifle_count": out[RIFLE_COLS_T].sum(axis=1),
            "ct_util_count": out[UTIL_COLS_CT].sum(axis=1),
            "t_util_count": out[UTIL_COLS_T].sum(axis=1),
            "ct_buy_tier": out["ct_money"].map(buy_tier),
            "t_buy_tier": out["t_money"].map(buy_tier),
            "target_cls": (out["round_winner"] == "T").astype(int),
        }
    )
    return pd.concat([out, new_cols], axis=1)


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_base = [
        "time_left", "ct_score", "t_score", "ct_health", "t_health",
        "ct_armor", "t_armor", "ct_money", "t_money", "ct_helmets", "t_helmets",
        "ct_defuse_kits", "ct_players_alive", "t_players_alive",
    ]
    derived = [
        "money_diff", "health_diff", "armor_diff", "players_alive_diff",
        "money_ratio", "ct_has_awp", "t_has_awp", "ct_rifle_count",
        "t_rifle_count", "ct_util_count", "t_util_count",
    ]
    weapon_cols = [c for c in df.columns if c.startswith(("ct_weapon_", "t_weapon_"))]
    grenade_cols = [c for c in df.columns if c.startswith(("ct_grenade_", "t_grenade_"))]
    numeric = numeric_base + derived + weapon_cols + grenade_cols + ["bomb_planted"]
    numeric = [c for c in numeric if c in df.columns]
    categorical = [c for c in ["map", "ct_buy_tier", "t_buy_tier"] if c in df.columns]
    return numeric, categorical

In [ ]:
df_raw = pd.read_csv(csv_path)
print("Bruto:", df_raw.shape)

df = filter_round_start(df_raw)
df = add_derived_features(df)
print("Após filtros:", df.shape)
print(f"target_cls (T): {df['target_cls'].mean():.1%}")
print("Mapas:", df["map"].value_counts().to_dict())

PROCESSED_CSV = DATA_PROCESSED / "rounds_start.csv"
df.to_csv(PROCESSED_CSV, index=False)

meta = {
    "rows": len(df),
    "columns": len(df.columns),
    "target_cls_balance_t_pct": round(df["target_cls"].mean() * 100, 2),
    "maps": df["map"].value_counts().to_dict(),
}
(DATA_PROCESSED / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print("Salvo:", PROCESSED_CSV)
print(json.dumps(meta, indent=2, ensure_ascii=False))

## 2. Pipeline sklearn (issue #6)

> **Limitação:** o dataset não traz ID de round — snapshots do mesmo round podem cair em treino e teste. Aceitável para o escopo da disciplina; em produção usaria agrupamento por partida/round.

In [ ]:
def make_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    numeric, categorical = get_feature_columns(df)
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
    )


numeric, categorical = get_feature_columns(df)
feature_cols = numeric + categorical
print(f"Features numéricas: {len(numeric)}")
print(f"Features categóricas: {len(categorical)}")

X_cls = df[feature_cols]
y_cls = df["target_cls"]
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
print(f"\nClassificação — treino: {X_train.shape}, teste: {X_test.shape}")
print(f"Balanceamento treino (T): {y_train.mean():.1%}")

In [ ]:
REGRESSION_EXCLUDE = {"money_diff", "money_ratio", "ct_money", "t_money"}
reg_cols = [c for c in feature_cols if c not in REGRESSION_EXCLUDE]
X_reg = df[reg_cols]
y_reg = df["money_diff"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print(f"Regressão — treino: {X_train_r.shape}, teste: {X_test_r.shape}")
print(f"money_diff médio (treino): {y_train_r.mean():.0f} USD")

In [ ]:
preprocessor = make_preprocessor(df)
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
print("Após pipeline (classificação):")
print(f"  treino: {X_train_t.shape}")
print(f"  teste:  {X_test_t.shape}")

---
Próximo passo: **`03_Modelagem.ipynb`** (issues #7–#10)

Variáveis disponíveis para o próximo notebook: `df`, `preprocessor`, `X_train`, `X_test`, `y_train`, `y_test`, splits de regressão.